In [1]:
# ============================================================
# CELL 1 — FINAL MODEL CONFIGURATION
# ============================================================

import time
import json
import numpy as np
import pandas as pd

from pathlib import Path

print("=" * 70)
print("FINAL MODEL — CONFIGURATION")
print("=" * 70)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path("..")

V2_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "engineered_features_v2.parquet"
)

OFFICIAL_SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "force2020_split.csv"
)

print("\nInput files:")
print(f"  V2 features : {V2_PATH}")
print(f"  Split file  : {OFFICIAL_SPLIT_PATH}")

assert V2_PATH.exists(), f"Missing V2 dataset: {V2_PATH}"
assert OFFICIAL_SPLIT_PATH.exists(), (
    f"Missing official split: {OFFICIAL_SPLIT_PATH}"
)

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

df_v2 = pd.read_parquet(V2_PATH)

official_split = pd.read_csv(
    OFFICIAL_SPLIT_PATH
)

print("\nLoaded:")
print(f"  V2 shape         : {df_v2.shape}")
print(f"  Unique wells     : {df_v2['WELL'].nunique()}")
print(f"  Split rows       : {len(official_split)}")
print(f"  Split wells      : {official_split['WELL'].nunique()}")

# ------------------------------------------------------------
# Official split audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OFFICIAL SPLIT")
print("=" * 70)

print(
    official_split["SPLIT"]
    .value_counts()
)

official_train_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "TRAIN",
        "WELL"
    ]
)

public_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "PUBLIC",
        "WELL"
    ]
)

blind_wells = set(
    official_split.loc[
        official_split["SPLIT"] == "BLIND_TEST",
        "WELL"
    ]
)

assert len(official_train_wells) == 98
assert len(public_wells) == 10
assert len(blind_wells) == 10

assert (
    official_train_wells.isdisjoint(public_wells)
)

assert (
    official_train_wells.isdisjoint(blind_wells)
)

assert (
    public_wells.isdisjoint(blind_wells)
)

print("\n✓ TRAIN      : 98 wells")
print("✓ PUBLIC     : 10 wells")
print("✓ BLIND_TEST : 10 wells")
print("✓ No split overlap")

# ------------------------------------------------------------
# Exact V2 + gradient feature set
# ------------------------------------------------------------

TARGET = "FORCE_2020_LITHOFACIES_LITHOLOGY"

BASE_FEATURES = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB",
]

MISSING_FEATURES = [
    "GR_MISSING",
    "RDEP_LOG10_MISSING",
    "RMED_LOG10_MISSING",
    "DTC_MISSING",
    "RHOB_MISSING",
]

ROLLING_FEATURES = []

for curve in [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB",
]:
    for window in [5, 9, 21]:
        ROLLING_FEATURES.extend([
            f"{curve}_ROLLMEAN_{window}",
            f"{curve}_ROLLSTD_{window}",
        ])

RESISTIVITY_FEATURES = [
    "RESISTIVITY_SEPARATION"
]

GRADIENT_FEATURES = [
    "GR_GRAD",
    "RDEP_LOG10_GRAD",
    "RMED_LOG10_GRAD",
    "DTC_GRAD",
    "RHOB_GRAD",
]

FINAL_FEATURES = (
    BASE_FEATURES
    + MISSING_FEATURES
    + ROLLING_FEATURES
    + RESISTIVITY_FEATURES
    + GRADIENT_FEATURES
)

print("\n" + "=" * 70)
print("FINAL FEATURE SET")
print("=" * 70)

print(f"\nBase features       : {len(BASE_FEATURES)}")
print(f"Missing indicators  : {len(MISSING_FEATURES)}")
print(f"Rolling features    : {len(ROLLING_FEATURES)}")
print(f"Resistivity feature : {len(RESISTIVITY_FEATURES)}")
print(f"Gradient features   : {len(GRADIENT_FEATURES)}")

print(f"\nTOTAL V2+GRAD FEATURES : {len(FINAL_FEATURES)}")

assert len(FINAL_FEATURES) == 46

# ------------------------------------------------------------
# Feature existence
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in FINAL_FEATURES
    if feature not in df_v2.columns
]

assert not missing_features, (
    f"Missing final features: {missing_features}"
)

assert TARGET in df_v2.columns
assert "WELL" in df_v2.columns

print("✓ All 46 final features exist")
print(f"✓ Target column exists: {TARGET}")

# ------------------------------------------------------------
# Check official TRAIN wells exist
# ------------------------------------------------------------

data_wells = set(
    df_v2["WELL"].dropna().unique()
)

missing_train_wells = (
    official_train_wells - data_wells
)

assert not missing_train_wells, (
    f"Official TRAIN wells missing from dataset: "
    f"{missing_train_wells}"
)

print("✓ All 98 official TRAIN wells are available")

# ------------------------------------------------------------
# Final configuration summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL MODEL CONFIGURATION FROZEN")
print("=" * 70)

print("""
Model candidate:
  XGBoost

Feature representation:
  V2 + gradients

Features:
  46

Training wells:
  98 official TRAIN wells

PUBLIC:
  10 wells — untouched

BLIND:
  10 wells — untouched

Development experiments:
  COMPLETE
""")

print("✓ CELL 1 COMPLETE")
print("=" * 70)

FINAL MODEL — CONFIGURATION

Input files:
  V2 features : ../data/processed/engineered_features_v2.parquet
  Split file  : ../data/splits/force2020_split.csv

Loaded:
  V2 shape         : (1430974, 52)
  Unique wells     : 118
  Split rows       : 118
  Split wells      : 118

OFFICIAL SPLIT
SPLIT
TRAIN         98
PUBLIC        10
BLIND_TEST    10
Name: count, dtype: int64

✓ TRAIN      : 98 wells
✓ PUBLIC     : 10 wells
✓ BLIND_TEST : 10 wells
✓ No split overlap

FINAL FEATURE SET

Base features       : 5
Missing indicators  : 5
Rolling features    : 30
Resistivity feature : 1
Gradient features   : 5

TOTAL V2+GRAD FEATURES : 46
✓ All 46 final features exist
✓ Target column exists: FORCE_2020_LITHOFACIES_LITHOLOGY
✓ All 98 official TRAIN wells are available

FINAL MODEL CONFIGURATION FROZEN

Model candidate:
  XGBoost

Feature representation:
  V2 + gradients

Features:
  46

Training wells:
  98 official TRAIN wells

PUBLIC:
  10 wells — untouched

BLIND:
  10 wells — untouched

Deve

In [2]:
# ============================================================
# CELL 2 — BUILD FINAL 98-WELL TRAINING DATASET
# ============================================================

print("=" * 70)
print("CELL 2 — FINAL 98-WELL TRAINING DATASET")
print("=" * 70)

# ------------------------------------------------------------
# 1. Keep ONLY official TRAIN wells
# ------------------------------------------------------------

df_final_train = df_v2[
    df_v2["WELL"].isin(official_train_wells)
].copy()

print("\nFinal training pool:")
print(f"  Rows  : {len(df_final_train):,}")
print(f"  Wells : {df_final_train['WELL'].nunique()}")

assert df_final_train["WELL"].nunique() == 98
assert len(
    set(df_final_train["WELL"]) & public_wells
) == 0
assert len(
    set(df_final_train["WELL"]) & blind_wells
) == 0

# ------------------------------------------------------------
# 2. Remove any stale split columns
# ------------------------------------------------------------

stale_split_columns = [
    col for col in df_final_train.columns
    if col in [
        "SPLIT",
        "DEV_SPLIT",
        "OFFICIAL_SPLIT"
    ]
]

if stale_split_columns:

    print(
        "\nRemoving stale split columns:",
        stale_split_columns
    )

    df_final_train = df_final_train.drop(
        columns=stale_split_columns
    )

# ------------------------------------------------------------
# 3. Construct final ML matrices
# ------------------------------------------------------------

X_final_train = df_final_train[
    FINAL_FEATURES
].copy()

y_final_train = df_final_train[
    TARGET
].copy()

print("\nFinal ML matrices:")
print(
    f"  X_final_train : "
    f"{X_final_train.shape}"
)

print(
    f"  y_final_train : "
    f"{y_final_train.shape}"
)

assert X_final_train.shape == (
    len(df_final_train),
    46
)

assert len(y_final_train) == len(
    X_final_train
)

# ------------------------------------------------------------
# 4. Verify all 12 lithology classes
# ------------------------------------------------------------

FINAL_CLASSES = np.array([
    30000.0,
    65000.0,
    65030.0,
    70000.0,
    70032.0,
    74000.0,
    80000.0,
    86000.0,
    88000.0,
    90000.0,
    93000.0,
    99000.0,
])

final_class_counts = (
    y_final_train
    .value_counts()
    .reindex(FINAL_CLASSES)
)

print("\n" + "=" * 70)
print("FINAL TRAINING CLASS DISTRIBUTION")
print("=" * 70)

display(
    final_class_counts
    .rename("samples")
    .to_frame()
)

assert final_class_counts.notna().all()
assert set(y_final_train.unique()) == set(
    FINAL_CLASSES
)

print("✓ All 12 classes present")

# ------------------------------------------------------------
# 5. Calculate final sqrt-balanced class weights
# ------------------------------------------------------------

N_FINAL = len(y_final_train)
K_FINAL = len(FINAL_CLASSES)

final_class_weights = np.sqrt(
    N_FINAL /
    (K_FINAL * final_class_counts)
)

print("\n" + "=" * 70)
print("FINAL SQRT-BALANCED CLASS WEIGHTS")
print("=" * 70)

for cls in FINAL_CLASSES:

    print(
        f"{int(cls):6d} : "
        f"count={int(final_class_counts[cls]):8,d}  "
        f"weight={final_class_weights[cls]:.6f}"
    )

# ------------------------------------------------------------
# 6. Convert class weights → sample weights
# ------------------------------------------------------------

final_sample_weights = (
    y_final_train
    .map(final_class_weights)
    .to_numpy(dtype=np.float64)
)

assert len(final_sample_weights) == len(
    y_final_train
)

assert np.isfinite(
    final_sample_weights
).all()

assert (
    final_sample_weights > 0
).all()

print("\n✓ Every training sample has a valid weight")

# ------------------------------------------------------------
# 7. Weight statistics
# ------------------------------------------------------------

print("\nFinal sample-weight statistics:")

print(
    f"  Minimum : "
    f"{final_sample_weights.min():.6f}"
)

print(
    f"  Maximum : "
    f"{final_sample_weights.max():.6f}"
)

print(
    f"  Mean    : "
    f"{final_sample_weights.mean():.6f}"
)

print(
    f"  Median  : "
    f"{np.median(final_sample_weights):.6f}"
)

# ------------------------------------------------------------
# 8. Final isolation audit
# ------------------------------------------------------------

training_wells = set(
    df_final_train["WELL"].unique()
)

assert training_wells == official_train_wells

assert training_wells.isdisjoint(
    public_wells
)

assert training_wells.isdisjoint(
    blind_wells
)

print("\n" + "=" * 70)
print("FINAL DATASET ISOLATION AUDIT")
print("=" * 70)

print("✓ Exactly 98 official TRAIN wells")
print("✓ 10 PUBLIC wells excluded")
print("✓ 10 BLIND_TEST wells excluded")
print("✓ 46 features")
print("✓ 12 lithology classes")
print("✓ Training-only sqrt-balanced weights")

print("\n" + "=" * 70)
print("✓ FINAL TRAINING DATASET FROZEN")
print("=" * 70)

CELL 2 — FINAL 98-WELL TRAINING DATASET

Final training pool:
  Rows  : 1,171,557
  Wells : 98

Removing stale split columns: ['SPLIT']

Final ML matrices:
  X_final_train : (1171557, 46)
  y_final_train : (1171557,)

FINAL TRAINING CLASS DISTRIBUTION


,samples
FORCE_2020_LITHOFACIES_LITHOLOGY,
30000.0,169405
65000.0,721185
65030.0,150605
70000.0,56320
70032.0,10513
74000.0,1688
80000.0,33336
86000.0,1085
88000.0,8214


✓ All 12 classes present

FINAL SQRT-BALANCED CLASS WEIGHTS
 30000 : count= 169,405  weight=0.759151
 65000 : count= 721,185  weight=0.367932
 65030 : count= 150,605  weight=0.805140
 70000 : count=  56,320  weight=1.316618
 70032 : count=  10,513  weight=3.047388
 74000 : count=   1,688  weight=7.605099
 80000 : count=  33,336  weight=1.711332
 86000 : count=   1,085  weight=9.485849
 88000 : count=   8,214  weight=3.447575
 90000 : count=   3,820  weight=5.055445
 93000 : count=     141  weight=26.313677
 99000 : count=  15,245  weight=2.530623

✓ Every training sample has a valid weight

Final sample-weight statistics:
  Minimum : 0.367932
  Maximum : 26.313677
  Mean    : 0.675594
  Median  : 0.367932

FINAL DATASET ISOLATION AUDIT
✓ Exactly 98 official TRAIN wells
✓ 10 PUBLIC wells excluded
✓ 10 BLIND_TEST wells excluded
✓ 46 features
✓ 12 lithology classes
✓ Training-only sqrt-balanced weights

✓ FINAL TRAINING DATASET FROZEN


In [10]:
# ============================================================
# DIAGNOSTIC — EXACT FINAL MODEL FEATURE COLUMNS
# ============================================================

print("=" * 70)
print("EXACT FEATURES USED BY FINAL TRAINING")
print("=" * 70)

# X_final_train is the matrix actually used to train the model
if hasattr(X_final_train, "columns"):
    final_feature_columns = list(X_final_train.columns)

    print(f"\nNumber of training features: {len(final_feature_columns)}")

    for i, col in enumerate(final_feature_columns, start=1):
        print(f"{i:2d}. {col}")
else:
    print("X_final_train does not have column names.")
    print("Type:", type(X_final_train))

print("\n" + "=" * 70)
print("FEATURE COLUMNS IN RELOADED V2 DATASET")
print("=" * 70)

v2_feature_candidates = [
    c for c in eval_df.columns
    if c not in ["WELL", "DEPT", target_column]
]

print(f"\nCandidate columns: {len(v2_feature_candidates)}")

for i, col in enumerate(v2_feature_candidates, start=1):
    print(f"{i:2d}. {col}")

print("\n" + "=" * 70)

EXACT FEATURES USED BY FINAL TRAINING

Number of training features: 46
 1. GR
 2. RDEP_LOG10
 3. RMED_LOG10
 4. DTC
 5. RHOB
 6. GR_MISSING
 7. RDEP_LOG10_MISSING
 8. RMED_LOG10_MISSING
 9. DTC_MISSING
10. RHOB_MISSING
11. GR_ROLLMEAN_5
12. GR_ROLLSTD_5
13. GR_ROLLMEAN_9
14. GR_ROLLSTD_9
15. GR_ROLLMEAN_21
16. GR_ROLLSTD_21
17. RDEP_LOG10_ROLLMEAN_5
18. RDEP_LOG10_ROLLSTD_5
19. RDEP_LOG10_ROLLMEAN_9
20. RDEP_LOG10_ROLLSTD_9
21. RDEP_LOG10_ROLLMEAN_21
22. RDEP_LOG10_ROLLSTD_21
23. RMED_LOG10_ROLLMEAN_5
24. RMED_LOG10_ROLLSTD_5
25. RMED_LOG10_ROLLMEAN_9
26. RMED_LOG10_ROLLSTD_9
27. RMED_LOG10_ROLLMEAN_21
28. RMED_LOG10_ROLLSTD_21
29. DTC_ROLLMEAN_5
30. DTC_ROLLSTD_5
31. DTC_ROLLMEAN_9
32. DTC_ROLLSTD_9
33. DTC_ROLLMEAN_21
34. DTC_ROLLSTD_21
35. RHOB_ROLLMEAN_5
36. RHOB_ROLLSTD_5
37. RHOB_ROLLMEAN_9
38. RHOB_ROLLSTD_9
39. RHOB_ROLLMEAN_21
40. RHOB_ROLLSTD_21
41. RESISTIVITY_SEPARATION
42. GR_GRAD
43. RDEP_LOG10_GRAD
44. RMED_LOG10_GRAD
45. DTC_GRAD
46. RHOB_GRAD

FEATURE COLUMNS IN RELOAD

In [11]:
# ============================================================
# CELL 4 — FINAL MODEL: PUBLIC SET EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

print("=" * 70)
print("CELL 4 — FINAL MODEL PUBLIC EVALUATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Reload official split
# ------------------------------------------------------------

split_df = pd.read_csv(
    "../data/splits/force2020_split.csv"
)

PUBLIC_WELLS = split_df.loc[
    split_df["SPLIT"] == "PUBLIC",
    "WELL"
].tolist()

print(f"\nOfficial PUBLIC wells: {len(PUBLIC_WELLS)}")

# ------------------------------------------------------------
# 2. Load V2 engineered dataset
# ------------------------------------------------------------

eval_df = pd.read_parquet(
    "../data/processed/engineered_features_v2.parquet"
)

print(f"V2 dataset shape: {eval_df.shape}")

# ------------------------------------------------------------
# 3. Use EXACT feature columns from final training
# ------------------------------------------------------------

final_feature_columns = list(X_final_train.columns)

print(f"\nFinal feature count: {len(final_feature_columns)}")

assert len(final_feature_columns) == 46

# Verify every training feature exists in V2
missing_features = [
    col for col in final_feature_columns
    if col not in eval_df.columns
]

assert not missing_features, (
    f"Missing final features: {missing_features}"
)

print("✓ All 46 final training features found")

# ------------------------------------------------------------
# 4. Select PUBLIC wells
# ------------------------------------------------------------

public_df = eval_df[
    eval_df["WELL"].isin(PUBLIC_WELLS)
].copy()

print("\nPUBLIC dataset:")
print(f"  Wells   : {public_df['WELL'].nunique()}")
print(f"  Samples : {len(public_df):,}")

assert public_df["WELL"].nunique() == 10

# ------------------------------------------------------------
# 5. Build PUBLIC feature matrix
# ------------------------------------------------------------

X_public = public_df[
    final_feature_columns
].copy()

y_public_force = public_df[
    target_column
].to_numpy()

print(f"  Features: {X_public.shape[1]}")

# Critical feature-order verification
assert list(X_public.columns) == final_feature_columns

print("✓ PUBLIC feature order exactly matches training")

# ------------------------------------------------------------
# 6. Encode PUBLIC target
# ------------------------------------------------------------

public_classes = np.sort(
    np.unique(y_public_force)
)

print("\nPUBLIC FORCE classes:")
print(public_classes)

unknown_classes = set(public_classes) - set(force_classes)

assert not unknown_classes, (
    f"Unexpected PUBLIC classes: {unknown_classes}"
)

y_public_encoded = np.array(
    [force_to_xgb[value] for value in y_public_force],
    dtype=np.int32
)

print("✓ PUBLIC target encoding verified")

# ------------------------------------------------------------
# 7. Predict PUBLIC
# ------------------------------------------------------------

print("\nPredicting PUBLIC...")

start_time = time.time()

public_pred_encoded = final_model.predict(
    X_public
)

prediction_time = time.time() - start_time

# Convert XGBoost indices back to FORCE codes
public_pred_force = np.array([
    xgb_to_force[int(value)]
    for value in public_pred_encoded
])

print("✓ PUBLIC prediction complete")
print(f"  Prediction time: {prediction_time:.2f} seconds")

# ------------------------------------------------------------
# 8. Calculate standard metrics
# ------------------------------------------------------------

public_accuracy = accuracy_score(
    y_public_force,
    public_pred_force
)

public_balanced_accuracy = balanced_accuracy_score(
    y_public_force,
    public_pred_force
)

public_macro_f1 = f1_score(
    y_public_force,
    public_pred_force,
    average="macro",
    labels=force_classes,
    zero_division=0
)

public_weighted_f1 = f1_score(
    y_public_force,
    public_pred_force,
    average="weighted",
    labels=force_classes,
    zero_division=0
)

print("\n" + "=" * 70)
print("PUBLIC METRICS")
print("=" * 70)

print(f"Accuracy          : {public_accuracy:.6f}")
print(f"Balanced Accuracy : {public_balanced_accuracy:.6f}")
print(f"Macro F1          : {public_macro_f1:.6f}")
print(f"Weighted F1       : {public_weighted_f1:.6f}")

# ------------------------------------------------------------
# 9. Classification report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PUBLIC CLASSIFICATION REPORT")
print("=" * 70)

public_report = classification_report(
    y_public_force,
    public_pred_force,
    labels=force_classes,
    zero_division=0
)

print(public_report)

print("\n" + "=" * 70)
print("PUBLIC EVALUATION COMPLETE")
print("=" * 70)

CELL 4 — FINAL MODEL PUBLIC EVALUATION

Official PUBLIC wells: 10
V2 dataset shape: (1430974, 52)

Final feature count: 46
✓ All 46 final training features found

PUBLIC dataset:
  Wells   : 10
  Samples : 136,841
  Features: 46
✓ PUBLIC feature order exactly matches training

PUBLIC FORCE classes:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 90000. 99000.]
✓ PUBLIC target encoding verified

Predicting PUBLIC...
✓ PUBLIC prediction complete
  Prediction time: 2.10 seconds

PUBLIC METRICS
Accuracy          : 0.708771
Balanced Accuracy : 0.472158
Macro F1          : 0.353070
Weighted F1       : 0.719331

PUBLIC CLASSIFICATION REPORT
              precision    recall  f1-score   support

     30000.0       0.80      0.79      0.79     24048
     65000.0       0.86      0.80      0.83     84030
     65030.0       0.36      0.32      0.33     17558
     70000.0       0.43      0.57      0.49      4798
     70032.0       0.32      0.06      0.10       625
     74000.0       0.02 

/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [12]:
# ============================================================
# CELL 5 — OFFICIAL FORCE SCORE ON PUBLIC
# ============================================================

print("=" * 70)
print("CELL 5 — OFFICIAL FORCE SCORE: PUBLIC")
print("=" * 70)

# ------------------------------------------------------------
# Official FORCE class order
# ------------------------------------------------------------

FORCE_CLASS_ORDER = [
    30000,  # Sandstone
    65030,  # Sandstone/Shale
    65000,  # Shale
    80000,  # Marl
    74000,  # Dolomite
    70000,  # Limestone
    70032,  # Chalk
    88000,  # Halite
    86000,  # Anhydrite
    99000,  # Tuff
    90000,  # Coal
    93000   # Basement
]

# ------------------------------------------------------------
# Official FORCE penalty matrix
# ------------------------------------------------------------

PENALTY_MATRIX = np.array([
    [0,    2,     3.5,   3,     3.75,  3.5,   3.5,   4,     4,     2.5,   3.875, 3.25],
    [2,    0,     2.375, 2.75,  4,     3.75,  3.75,  3.875, 4,     3,     3.75,  3],
    [3.5,  2.375, 0,     2,     3.5,   3.5,   3.75,  4,     3.25,  2.75,  3.25,  3],
    [3,    2.75,  2,     0,     2.5,   2,     2.25,  4,     4,     3.375, 3.75,  3.25],
    [3.75, 4,     3.5,   2.5,   0,     2.625, 2.875, 3.75,  3,     3,     4,     3.625],
    [3.5,  3.75,  3.5,   2,     2.625, 0,     1.375, 4,     3.75,  3.5,   4,     3.625],
    [3.5,  3.75,  3.75,  2.25,  2.875, 1.375, 0,     4,     3.75,  3.125, 4,     3.75],
    [4,    3.875, 4,     4,     3.75,  4,     4,     4,     2.75,  3.75,  3.75,  4],
    [4,    4,     4,     4,     3.25, 3.75, 3.75,  2.75,  0,     4,     4,     3.875],
    [2.5,  3,     2.75,  3.375, 3,     3.5,   3.125, 3.75,  4,     0,     2.5,   3.25],
    [3.875,3.75,  3.25,   3.75,  4,     4,     4,     3.75,  4,     2.5,   0,     4],
    [3.25, 3,     3,      3.25,  3.625, 3.625, 3.75, 4,     3.875, 3.25,  4,     0]
])

# ------------------------------------------------------------
# Create FORCE-code -> official matrix index mapping
# ------------------------------------------------------------

force_index = {
    code: idx
    for idx, code in enumerate(FORCE_CLASS_ORDER)
}

# ------------------------------------------------------------
# Convert true/predicted FORCE codes to matrix indices
# ------------------------------------------------------------

true_idx = np.array([
    force_index[int(code)]
    for code in y_public_force
])

pred_idx = np.array([
    force_index[int(code)]
    for code in public_pred_force
])

# ------------------------------------------------------------
# Calculate per-sample penalties
# ------------------------------------------------------------

sample_penalties = PENALTY_MATRIX[
    true_idx,
    pred_idx
]

mean_penalty = np.mean(sample_penalties)

public_force_score = -mean_penalty

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nOfficial FORCE evaluation:")
print(f"  Samples       : {len(sample_penalties):,}")
print(f"  Mean penalty  : {mean_penalty:.6f}")
print(f"  FORCE score   : {public_force_score:.6f}")

print("\nPrevious W5 BLIND benchmark:")
print("  FORCE score   : -0.687657")

difference = public_force_score - (-0.687657)

print(f"\nDifference vs W5 benchmark: {difference:+.6f}")

if public_force_score > -0.687657:
    print("✓ Final XGBoost is better than the W5 benchmark on PUBLIC")
elif public_force_score < -0.687657:
    print("✗ Final XGBoost is worse than the W5 benchmark on PUBLIC")
else:
    print("= Final XGBoost matches the W5 benchmark")

print("\n" + "=" * 70)
print("PUBLIC FORCE SCORE COMPLETE")
print("=" * 70)

CELL 5 — OFFICIAL FORCE SCORE: PUBLIC

Official FORCE evaluation:
  Samples       : 136,841
  Mean penalty  : 0.734295
  FORCE score   : -0.734295

Previous W5 BLIND benchmark:
  FORCE score   : -0.687657

Difference vs W5 benchmark: -0.046638
✗ Final XGBoost is worse than the W5 benchmark on PUBLIC

PUBLIC FORCE SCORE COMPLETE


In [6]:
# ------------------------------------------------------------
# 3. Train final model on all 98 official TRAIN wells
# ------------------------------------------------------------

print("\nTraining final model...")
print(f"  Samples : {X_final_train.shape[0]:,}")
print(f"  Features: {X_final_train.shape[1]}")
print(f"  Wells   : 98")

start_time = time.time()

final_model.fit(
    X_final_train,
    y_final_train_encoded,
    sample_weight=final_sample_weights
)

training_time = time.time() - start_time

print(f"\n✓ Final model trained successfully")
print(f"  Training time: {training_time:.2f} seconds")

# ------------------------------------------------------------
# 4. Verify trained model
# ------------------------------------------------------------

print("\nModel verification:")
print(f"  Number of classes: {final_model.n_classes_}")
print(f"  Model classes    : {final_model.classes_}")

assert final_model.n_classes_ == 12
assert np.array_equal(
    final_model.classes_,
    np.arange(12)
)

print("✓ Model class encoding verified")

print("\n" + "=" * 70)
print("FINAL TRAINING COMPLETE")
print("=" * 70)


Training final model...
  Samples : 1,171,557
  Features: 46
  Wells   : 98

✓ Final model trained successfully
  Training time: 201.96 seconds

Model verification:
  Number of classes: 12
  Model classes    : [ 0  1  2  3  4  5  6  7  8  9 10 11]
✓ Model class encoding verified

FINAL TRAINING COMPLETE


In [13]:
# ============================================================
# CELL 6 — FINAL MODEL: BLIND TEST PREDICTION
# ============================================================

print("=" * 70)
print("CELL 6 — FINAL MODEL BLIND TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Identify official BLIND wells
# ------------------------------------------------------------

BLIND_WELLS = split_df.loc[
    split_df["SPLIT"] == "BLIND_TEST",
    "WELL"
].tolist()

print(f"\nOfficial BLIND wells: {len(BLIND_WELLS)}")

# ------------------------------------------------------------
# 2. Select BLIND wells from V2 dataset
# ------------------------------------------------------------

blind_df = eval_df[
    eval_df["WELL"].isin(BLIND_WELLS)
].copy()

print("\nBLIND dataset:")
print(f"  Wells   : {blind_df['WELL'].nunique()}")
print(f"  Samples : {len(blind_df):,}")

assert blind_df["WELL"].nunique() == 10

# ------------------------------------------------------------
# 3. Build exact final feature matrix
# ------------------------------------------------------------

X_blind = blind_df[
    final_feature_columns
].copy()

y_blind_force = blind_df[
    target_column
].to_numpy()

print(f"  Features: {X_blind.shape[1]}")

# Critical feature-order check
assert list(X_blind.columns) == final_feature_columns

print("✓ BLIND feature order exactly matches training")

# ------------------------------------------------------------
# 4. Verify all blind targets are available
# ------------------------------------------------------------

blind_classes = np.sort(
    np.unique(y_blind_force)
)

print("\nBLIND FORCE classes:")
print(blind_classes)

print(f"  Number of classes present: {len(blind_classes)}")

# ------------------------------------------------------------
# 5. Predict BLIND
# ------------------------------------------------------------

print("\nPredicting BLIND...")
print("IMPORTANT: model is frozen — no retraining or tuning")

start_time = time.time()

blind_pred_encoded = final_model.predict(
    X_blind
)

prediction_time = time.time() - start_time

# Convert XGBoost class indices back to FORCE codes
blind_pred_force = np.array([
    xgb_to_force[int(value)]
    for value in blind_pred_encoded
])

print("\n✓ BLIND prediction complete")
print(f"  Prediction time: {prediction_time:.2f} seconds")

# ------------------------------------------------------------
# 6. Basic prediction audit
# ------------------------------------------------------------

print("\nPrediction distribution:")

unique_pred, pred_counts = np.unique(
    blind_pred_force,
    return_counts=True
)

for code, count in zip(unique_pred, pred_counts):
    percentage = 100 * count / len(blind_pred_force)
    print(
        f"  {int(code):5d}: "
        f"{count:7,d} samples "
        f"({percentage:6.2f}%)"
    )

print("\n" + "=" * 70)
print("BLIND PREDICTION COMPLETE")
print("=" * 70)

CELL 6 — FINAL MODEL BLIND TEST

Official BLIND wells: 10

BLIND dataset:
  Wells   : 10
  Samples : 122,576
  Features: 46
✓ BLIND feature order exactly matches training

BLIND FORCE classes:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 88000. 90000.
 99000.]
  Number of classes present: 11

Predicting BLIND...
IMPORTANT: model is frozen — no retraining or tuning

✓ BLIND prediction complete
  Prediction time: 1.12 seconds

Prediction distribution:
  30000:  11,597 samples (  9.46%)
  65000:  75,564 samples ( 61.65%)
  65030:   9,292 samples (  7.58%)
  70000:   7,927 samples (  6.47%)
  70032:   2,315 samples (  1.89%)
  74000:     396 samples (  0.32%)
  80000:   6,355 samples (  5.18%)
  86000:     480 samples (  0.39%)
  88000:   5,989 samples (  4.89%)
  90000:     423 samples (  0.35%)
  99000:   2,238 samples (  1.83%)

BLIND PREDICTION COMPLETE


In [14]:
# ============================================================
# CELL 7 — OFFICIAL FORCE SCORE ON BLIND
# ============================================================

print("=" * 70)
print("CELL 7 — OFFICIAL FORCE SCORE: BLIND")
print("=" * 70)

# ------------------------------------------------------------
# Official FORCE class order
# ------------------------------------------------------------

FORCE_CLASS_ORDER = [
    30000,  # Sandstone
    65030,  # Sandstone/Shale
    65000,  # Shale
    80000,  # Marl
    74000,  # Dolomite
    70000,  # Limestone
    70032,  # Chalk
    88000,  # Halite
    86000,  # Anhydrite
    99000,  # Tuff
    90000,  # Coal
    93000   # Basement
]

# ------------------------------------------------------------
# Official FORCE penalty matrix
# ------------------------------------------------------------

PENALTY_MATRIX = np.array([
    [0,    2,     3.5,   3,     3.75,  3.5,   3.5,   4,     4,     2.5,   3.875, 3.25],
    [2,    0,     2.375, 2.75,  4,     3.75,  3.75,  3.875, 4,     3,     3.75,  3],
    [3.5,  2.375, 0,     2,     3.5,   3.5,   3.75,  4,     3.25,  2.75,  3.25,  3],
    [3,    2.75,  2,     0,     2.5,   2,     2.25,  4,     4,     3.375, 3.75,  3.25],
    [3.75, 4,     3.5,   2.5,   0,     2.625, 2.875, 3.75,  3,     3,     4,     3.625],
    [3.5,  3.75,  3.5,   2,     2.625, 0,     1.375, 4,     3.75,  3.5,   4,     3.625],
    [3.5,  3.75,  3.75,  2.25,  2.875, 1.375, 0,     4,     3.75,  3.125, 4,     3.75],
    [4,    3.875, 4,     4,     3.75,  4,     4,     4,     2.75,  3.75,  3.75,  4],
    [4,    4,     4,     4,     3.25, 3.75, 3.75,  2.75,  0,     4,     4,     3.875],
    [2.5,  3,     2.75,  3.375, 3,     3.5,   3.125, 3.75,  4,     0,     2.5,   3.25],
    [3.875, 3.75, 3.25, 3.75,  4,     4,     4,     3.75,  4,     2.5,   0,     4],
    [3.25, 3,     3,     3.25,  3.625, 3.625, 3.75,  4,     3.875, 3.25,  4,     0]
])

# ------------------------------------------------------------
# FORCE code -> penalty-matrix index
# ------------------------------------------------------------

force_index = {
    code: idx
    for idx, code in enumerate(FORCE_CLASS_ORDER)
}

# ------------------------------------------------------------
# Convert true/predicted labels to matrix indices
# ------------------------------------------------------------

true_idx = np.array([
    force_index[int(code)]
    for code in y_blind_force
])

pred_idx = np.array([
    force_index[int(code)]
    for code in blind_pred_force
])

# ------------------------------------------------------------
# Calculate sample-wise geological penalties
# ------------------------------------------------------------

sample_penalties = PENALTY_MATRIX[
    true_idx,
    pred_idx
]

mean_penalty = np.mean(sample_penalties)

blind_force_score = -mean_penalty

# ------------------------------------------------------------
# Compare against W5 blind benchmark
# ------------------------------------------------------------

W5_BLIND_SCORE = -0.687657

difference = blind_force_score - W5_BLIND_SCORE

print("\nOfficial BLIND evaluation:")
print(f"  Samples      : {len(sample_penalties):,}")
print(f"  Mean penalty : {mean_penalty:.6f}")
print(f"  FORCE score  : {blind_force_score:.6f}")

print("\nPrevious W5 BLIND benchmark:")
print(f"  FORCE score  : {W5_BLIND_SCORE:.6f}")

print(f"\nDifference vs W5: {difference:+.6f}")

if blind_force_score > W5_BLIND_SCORE:
    print("✓ FINAL XGBOOST BEATS W5")
elif blind_force_score < W5_BLIND_SCORE:
    print("✗ FINAL XGBOOST DOES NOT BEAT W5")
else:
    print("= FINAL XGBOOST MATCHES W5")

print("\n" + "=" * 70)
print("BLIND FORCE SCORE COMPLETE")
print("=" * 70)


CELL 7 — OFFICIAL FORCE SCORE: BLIND

Official BLIND evaluation:
  Samples      : 122,576
  Mean penalty : 0.961986
  FORCE score  : -0.961986

Previous W5 BLIND benchmark:
  FORCE score  : -0.687657

Difference vs W5: -0.274329
✗ FINAL XGBOOST DOES NOT BEAT W5

BLIND FORCE SCORE COMPLETE


In [15]:
# ============================================================
# CELL 8 — BLIND ERROR + GEOLOGICAL PENALTY AUDIT
# ============================================================

from sklearn.metrics import confusion_matrix
import pandas as pd

print("=" * 70)
print("CELL 8 — BLIND ERROR + GEOLOGICAL PENALTY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    y_blind_force,
    blind_pred_force,
    labels=FORCE_CLASS_ORDER
)

cm_df = pd.DataFrame(
    cm,
    index=FORCE_CLASS_ORDER,
    columns=FORCE_CLASS_ORDER
)

print("\nCONFUSION MATRIX")
print("(rows = TRUE, columns = PREDICTED)")
print(cm_df)

# ------------------------------------------------------------
# 2. True-class recall
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRUE-CLASS RECALL")
print("=" * 70)

for i, code in enumerate(FORCE_CLASS_ORDER):

    true_count = cm[i].sum()

    if true_count == 0:
        recall = np.nan
    else:
        recall = cm[i, i] / true_count

    print(
        f"{code:5d} | "
        f"true={true_count:7,d} | "
        f"correct={cm[i,i]:7,d} | "
        f"recall={recall:.4f}"
    )

# ------------------------------------------------------------
# 3. Predicted-class distribution and precision
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PREDICTED-CLASS PRECISION")
print("=" * 70)

for j, code in enumerate(FORCE_CLASS_ORDER):

    predicted_count = cm[:, j].sum()

    if predicted_count == 0:
        precision = np.nan
    else:
        precision = cm[j, j] / predicted_count

    print(
        f"{code:5d} | "
        f"predicted={predicted_count:7,d} | "
        f"correct={cm[j,j]:7,d} | "
        f"precision={precision:.4f}"
    )

# ------------------------------------------------------------
# 4. Most common WRONG predictions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 20 WRONG TRUE → PREDICTED PAIRS")
print("=" * 70)

wrong_pairs = []

for i, true_code in enumerate(FORCE_CLASS_ORDER):
    for j, pred_code in enumerate(FORCE_CLASS_ORDER):

        if i == j:
            continue

        count = cm[i, j]

        if count > 0:
            penalty = PENALTY_MATRIX[i, j]
            total_penalty = count * penalty

            wrong_pairs.append({
                "TRUE": true_code,
                "PRED": pred_code,
                "COUNT": count,
                "PERCENT_OF_BLIND": 100 * count / len(y_blind_force),
                "PENALTY_PER_SAMPLE": penalty,
                "TOTAL_PENALTY": total_penalty
            })

wrong_pairs_df = pd.DataFrame(wrong_pairs)

wrong_pairs_df = wrong_pairs_df.sort_values(
    "COUNT",
    ascending=False
)

print(
    wrong_pairs_df.head(20).to_string(
        index=False,
        formatters={
            "PERCENT_OF_BLIND": "{:.3f}".format,
            "PENALTY_PER_SAMPLE": "{:.3f}".format,
            "TOTAL_PENALTY": "{:.1f}".format
        }
    )
)

# ------------------------------------------------------------
# 5. Largest contributors to geological penalty
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOP 20 GEOLOGICAL PENALTY CONTRIBUTORS")
print("=" * 70)

penalty_pairs_df = wrong_pairs_df.sort_values(
    "TOTAL_PENALTY",
    ascending=False
).copy()

total_penalty = sample_penalties.sum()

penalty_pairs_df["PERCENT_OF_TOTAL_PENALTY"] = (
    100
    * penalty_pairs_df["TOTAL_PENALTY"]
    / total_penalty
)

print(
    penalty_pairs_df.head(20).to_string(
        index=False,
        formatters={
            "PERCENT_OF_BLIND": "{:.3f}".format,
            "PENALTY_PER_SAMPLE": "{:.3f}".format,
            "TOTAL_PENALTY": "{:.1f}".format,
            "PERCENT_OF_TOTAL_PENALTY": "{:.2f}".format
        }
    )
)

# ------------------------------------------------------------
# 6. Specifically investigate Halite predictions
# ------------------------------------------------------------

HALITE = 88000

halite_pred_mask = blind_pred_force == HALITE

halite_pred_count = halite_pred_mask.sum()

halite_correct = (
    halite_pred_mask &
    (y_blind_force == HALITE)
).sum()

halite_wrong = halite_pred_count - halite_correct

print("\n" + "=" * 70)
print("HALITE (88000) AUDIT")
print("=" * 70)

print(f"Predicted Halite : {halite_pred_count:,}")
print(f"Correct Halite   : {halite_correct:,}")
print(f"Wrong Halite     : {halite_wrong:,}")

if halite_pred_count > 0:
    print(
        f"Halite precision : "
        f"{halite_correct / halite_pred_count:.4f}"
    )

# What were those samples actually?
if halite_wrong > 0:

    actual_when_halite_predicted = (
        y_blind_force[halite_pred_mask]
    )

    actual_counts = pd.Series(
        actual_when_halite_predicted
    ).value_counts()

    print("\nTRUE classes when model predicted Halite:")
    print(actual_counts.to_string())

# ------------------------------------------------------------
# 7. Overall penalty accounting
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OVERALL PENALTY ACCOUNTING")
print("=" * 70)

print(f"Total penalty : {total_penalty:.3f}")
print(f"Mean penalty  : {mean_penalty:.6f}")
print(f"FORCE score   : {-mean_penalty:.6f}")

print("\n" + "=" * 70)
print("BLIND ERROR AUDIT COMPLETE")
print("=" * 70)

CELL 8 — BLIND ERROR + GEOLOGICAL PENALTY AUDIT

CONFUSION MATRIX
(rows = TRUE, columns = PREDICTED)
       30000  65030  65000  80000  74000  70000  70032  88000  86000  99000  \
30000   6825   1891   2933   1065     49    672    250      0      4    508   
65030   1607   2770   6586    700     58    174      2      0      0    332   
65000   1220   3597  63207   2436    120    489      0      0      3    641   
80000    136    636   1572   1143      2    716     99      0      0     76   
74000     12      0    102     12     48     17      8      0     80      8   
70000   1282    369    751    862     89   4584    314      0     39     82   
70032    142      0      0      1      0    768   1634      0      0    360   
88000    221      0      1      1      0    334      8   5933      0      0   
86000      0     12      0     13     27    135      0     56    354      0   
99000    135     15    398    122      3     37      0      0      0    231   
90000     17      2     14    

| True → Predicted | Samples | Penalty contribution |
|---|---:|---:|
| 65030 → 65000 | 6,586 | **13.27%** |
| 30000 → 65000 | 2,933 | **8.71%** |
| 65000 → 65030 | 3,597 | **7.24%** |
| 65000 → 80000 | 2,436 | 4.13% |
| 70000 → 30000 | 1,282 | 3.81% |
| 65000 → 30000 | 1,220 | 3.62% |

In [16]:
# ============================================================
# COMPARISON — CHECK W5 MODEL AVAILABILITY
# ============================================================

print("=" * 70)
print("CHECKING W5 MODEL")
print("=" * 70)

print("\nVariables currently available:")

for name in [
    "w5_model",
    "final_model",
    "X_final_train",
    "X_blind",
    "y_blind_force",
    "blind_pred_force"
]:
    print(f"  {name:20s}: {'YES' if name in globals() else 'NO'}")

print("\n" + "=" * 70)

CHECKING W5 MODEL

Variables currently available:
  w5_model            : NO
  final_model         : YES
  X_final_train       : YES
  X_blind             : YES
  y_blind_force       : YES
  blind_pred_force    : YES



In [17]:
# ============================================================
# CELL 9 — REBUILD W5 EXACTLY
# ============================================================

from sklearn.ensemble import ExtraTreesClassifier

print("=" * 70)
print("CELL 9 — REBUILD W5 EXTRA TREES")
print("=" * 70)

# ------------------------------------------------------------
# 1. Load engineered feature dataset
# ------------------------------------------------------------

w5_df = pd.read_parquet(
    "../data/processed/engineered_features.parquet"
)

print("\nEngineered dataset:")
print(f"  Shape: {w5_df.shape}")

# ------------------------------------------------------------
# 2. Official TRAIN wells
# ------------------------------------------------------------

TRAIN_WELLS = split_df.loc[
    split_df["SPLIT"] == "TRAIN",
    "WELL"
].tolist()

print(f"\nOfficial TRAIN wells: {len(TRAIN_WELLS)}")

assert len(TRAIN_WELLS) == 98

# ------------------------------------------------------------
# 3. Exact W5 feature set
# ------------------------------------------------------------

w5_base_features = [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB"
]

w5_missing_features = [
    "GR_MISSING",
    "RDEP_LOG10_MISSING",
    "RMED_LOG10_MISSING",
    "DTC_MISSING",
    "RHOB_MISSING"
]

w5_rolling_features = []

for curve in w5_base_features:
    for window in [5, 9, 21]:
        w5_rolling_features.extend([
            f"{curve}_ROLLMEAN_{window}",
            f"{curve}_ROLLSTD_{window}"
        ])

w5_features = (
    w5_base_features
    + w5_missing_features
    + w5_rolling_features
    + ["RESISTIVITY_SEPARATION"]
)

print(f"\nW5 feature count: {len(w5_features)}")

assert len(w5_features) == 41

# ------------------------------------------------------------
# 4. Verify features exist
# ------------------------------------------------------------

missing_w5_features = [
    col for col in w5_features
    if col not in w5_df.columns
]

assert not missing_w5_features, (
    f"Missing W5 features: {missing_w5_features}"
)

print("✓ All 41 W5 features found")

# ------------------------------------------------------------
# 5. Select official TRAIN data
# ------------------------------------------------------------

w5_train_df = w5_df[
    w5_df["WELL"].isin(TRAIN_WELLS)
].copy()

X_w5_train = w5_train_df[
    w5_features
].copy()

y_w5_train_force = w5_train_df[
    target_column
].to_numpy()

print("\nW5 training data:")
print(f"  Wells   : {w5_train_df['WELL'].nunique()}")
print(f"  Samples : {len(w5_train_df):,}")
print(f"  Features: {X_w5_train.shape[1]}")

assert w5_train_df["WELL"].nunique() == 98
assert X_w5_train.shape[1] == 41

# ------------------------------------------------------------
# 6. Encode target using same mapping
# ------------------------------------------------------------

w5_train_classes = np.sort(
    np.unique(y_w5_train_force)
)

assert np.array_equal(
    w5_train_classes,
    force_classes
)

y_w5_train_encoded = np.array([
    force_to_xgb[value]
    for value in y_w5_train_force
], dtype=np.int32)

# ------------------------------------------------------------
# 7. Recalculate EXACT sqrt-balanced weights
# ------------------------------------------------------------

class_counts = pd.Series(
    y_w5_train_encoded
).value_counts().sort_index()

n_samples = len(y_w5_train_encoded)
n_classes = len(force_classes)

sqrt_weights = np.sqrt(
    n_samples / (n_classes * class_counts)
)

w5_weight_map = sqrt_weights.to_dict()

w5_sample_weights = np.array([
    w5_weight_map[int(label)]
    for label in y_w5_train_encoded
])

print("\nW5 weight range:")
print(f"  Min: {w5_sample_weights.min():.6f}")
print(f"  Max: {w5_sample_weights.max():.6f}")

# ------------------------------------------------------------
# 8. Train EXACT W5 model
# ------------------------------------------------------------

w5_model = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=5,
    class_weight=None,
    n_jobs=-1,
    random_state=42
)

print("\nTraining W5...")

start_time = time.time()

w5_model.fit(
    X_w5_train,
    y_w5_train_encoded,
    sample_weight=w5_sample_weights
)

w5_training_time = time.time() - start_time

print(f"\n✓ W5 rebuilt successfully")
print(f"  Training time: {w5_training_time:.2f} seconds")

print("\n" + "=" * 70)
print("W5 REBUILD COMPLETE")
print("=" * 70)

CELL 9 — REBUILD W5 EXTRA TREES

Engineered dataset:
  Shape: (1430974, 46)

Official TRAIN wells: 98

W5 feature count: 41
✓ All 41 W5 features found

W5 training data:
  Wells   : 98
  Samples : 1,171,557
  Features: 41

W5 weight range:
  Min: 0.367932
  Max: 26.313677

Training W5...

✓ W5 rebuilt successfully
  Training time: 37.33 seconds

W5 REBUILD COMPLETE


In [18]:
# ============================================================
# CHECK FOR SAVED W5 MODEL
# ============================================================

from pathlib import Path

project_root = Path("..")

print("=" * 70)
print("SEARCHING FOR SAVED MODEL FILES")
print("=" * 70)

for path in project_root.rglob("*"):
    if path.is_file():
        name = path.name.lower()

        if any(key in name for key in [
            "w5",
            "extratree",
            "extra_tree",
            "model"
        ]):
            print(path)

print("\n" + "=" * 70)

SEARCHING FOR SAVED MODEL FILES
../models/w5_feature_metadata.json
../models/lithology_w5_final.joblib
../models/w5_class_weights.json
../notebooks/06_model_baseline.ipynb
../notebooks/07_final_model.ipynb
../notebooks/12_final_models_exp.ipynb
../notebooks/11_class_specific_model.ipynb
../notebooks/06experimetal_model_baseline.ipynb
../.venv/lib/python3.13/site-packages/httpcore/_models.py
../.venv/lib/python3.13/site-packages/pygments/modeline.py
../.venv/lib/python3.13/site-packages/cffi/model.py
../.venv/lib/python3.13/site-packages/charset_normalizer/models.py
../.venv/lib/python3.13/site-packages/requests/models.py
../.venv/lib/python3.13/site-packages/httpx/_models.py
../.venv/lib/python3.13/site-packages/catboost/eval/_fold_model.py
../.venv/lib/python3.13/site-packages/catboost/eval/_fold_models_handler.py
../.venv/lib/python3.13/site-packages/catboost/eval/__pycache__/_fold_model.cpython-313.pyc
../.venv/lib/python3.13/site-packages/catboost/eval/__pycache__/_fold_models_hand

In [19]:
# ============================================================
# LOAD SAVED W5 PRODUCTION MODEL
# ============================================================

import joblib

print("=" * 70)
print("LOADING SAVED W5 PRODUCTION MODEL")
print("=" * 70)

W5_MODEL_PATH = "../models/lithology_w5_final.joblib"

w5_saved_model = joblib.load(W5_MODEL_PATH)

print("\n✓ Saved W5 model loaded")
print(f"Model type: {type(w5_saved_model).__name__}")

print("\nModel configuration:")
print(f"  n_estimators     : {w5_saved_model.n_estimators}")
print(f"  max_depth        : {w5_saved_model.max_depth}")
print(f"  min_samples_leaf : {w5_saved_model.min_samples_leaf}")
print(f"  class_weight     : {w5_saved_model.class_weight}")
print(f"  n_jobs            : {w5_saved_model.n_jobs}")
print(f"  random_state      : {w5_saved_model.random_state}")

print("\n" + "=" * 70)

LOADING SAVED W5 PRODUCTION MODEL

✓ Saved W5 model loaded
Model type: ExtraTreesClassifier

Model configuration:
  n_estimators     : 100
  max_depth        : 20
  min_samples_leaf : 5
  class_weight     : None
  n_jobs            : -1
  random_state      : 42



In [21]:
# ============================================================
# DIAGNOSTIC — SAVED W5 FEATURE ORDER
# ============================================================

print("=" * 70)
print("SAVED W5 FEATURE ORDER")
print("=" * 70)

if hasattr(w5_saved_model, "feature_names_in_"):

    w5_saved_feature_columns = list(
        w5_saved_model.feature_names_in_
    )

    print(
        f"\nSaved W5 expects "
        f"{len(w5_saved_feature_columns)} features:"
    )

    for i, col in enumerate(
        w5_saved_feature_columns,
        start=1
    ):
        print(f"{i:2d}. {col}")

else:
    print("Saved W5 does not contain feature_names_in_.")

print("\nCurrent W5 feature order:")
for i, col in enumerate(w5_features, start=1):
    print(f"{i:2d}. {col}")

print("\n" + "=" * 70)

SAVED W5 FEATURE ORDER

Saved W5 expects 41 features:
 1. GR
 2. RDEP_LOG10
 3. RMED_LOG10
 4. DTC
 5. RHOB
 6. GR_ROLLMEAN_5
 7. GR_ROLLSTD_5
 8. GR_ROLLMEAN_9
 9. GR_ROLLSTD_9
10. GR_ROLLMEAN_21
11. GR_ROLLSTD_21
12. RDEP_LOG10_ROLLMEAN_5
13. RDEP_LOG10_ROLLSTD_5
14. RDEP_LOG10_ROLLMEAN_9
15. RDEP_LOG10_ROLLSTD_9
16. RDEP_LOG10_ROLLMEAN_21
17. RDEP_LOG10_ROLLSTD_21
18. RMED_LOG10_ROLLMEAN_5
19. RMED_LOG10_ROLLSTD_5
20. RMED_LOG10_ROLLMEAN_9
21. RMED_LOG10_ROLLSTD_9
22. RMED_LOG10_ROLLMEAN_21
23. RMED_LOG10_ROLLSTD_21
24. DTC_ROLLMEAN_5
25. DTC_ROLLSTD_5
26. DTC_ROLLMEAN_9
27. DTC_ROLLSTD_9
28. DTC_ROLLMEAN_21
29. DTC_ROLLSTD_21
30. RHOB_ROLLMEAN_5
31. RHOB_ROLLSTD_5
32. RHOB_ROLLMEAN_9
33. RHOB_ROLLSTD_9
34. RHOB_ROLLMEAN_21
35. RHOB_ROLLSTD_21
36. RESISTIVITY_SEPARATION
37. GR_MISSING
38. RDEP_LOG10_MISSING
39. RMED_LOG10_MISSING
40. DTC_MISSING
41. RHOB_MISSING

Current W5 feature order:
 1. GR
 2. RDEP_LOG10
 3. RMED_LOG10
 4. DTC
 5. RHOB
 6. GR_MISSING
 7. RDEP_LOG10_MISSING
 8.

In [22]:
# ============================================================
# PREDICT BLIND WITH SAVED W5 — EXACT FEATURE ORDER
# ============================================================

import time
import numpy as np

# Use EXACT feature order stored inside the saved model
w5_saved_feature_columns = list(
    w5_saved_model.feature_names_in_
)

X_w5_blind = blind_df[w5_saved_feature_columns]

print("BLIND matrix:", X_w5_blind.shape)
print("Expected:", (122576, 41))

assert X_w5_blind.shape == (122576, 41)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------
print("\nPredicting BLIND with saved W5...")

start_time = time.time()

w5_blind_pred_raw = w5_saved_model.predict(
    X_w5_blind
)

prediction_time = time.time() - start_time

print(f"Prediction time: {prediction_time:.2f} s")
print("Predictions:", len(w5_blind_pred_raw))

# ------------------------------------------------------------
# Inspect labels BEFORE any conversion
# ------------------------------------------------------------
print("\nSaved W5 model classes:")
print(w5_saved_model.classes_)

print("\nUnique predicted labels:")
print(np.unique(w5_blind_pred_raw))

print("\nPrediction distribution:")
unique, counts = np.unique(
    w5_blind_pred_raw,
    return_counts=True
)

for label, count in zip(unique, counts):
    print(f"{label}: {count:,} ({count / len(w5_blind_pred_raw):.2%})")

BLIND matrix: (122576, 41)
Expected: (122576, 41)

Predicting BLIND with saved W5...
Prediction time: 0.55 s
Predictions: 122576

Saved W5 model classes:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 88000. 90000.
 93000. 99000.]

Unique predicted labels:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 88000. 90000.
 93000. 99000.]

Prediction distribution:
30000.0: 12,456 (10.16%)
65000.0: 83,543 (68.16%)
65030.0: 7,354 (6.00%)
70000.0: 7,373 (6.02%)
70032.0: 1,125 (0.92%)
74000.0: 25 (0.02%)
80000.0: 3,294 (2.69%)
86000.0: 548 (0.45%)
88000.0: 5,817 (4.75%)
90000.0: 382 (0.31%)
93000.0: 4 (0.00%)
99000.0: 655 (0.53%)


In [25]:
blind_df

,GR,RDEP,RMED,DTC,RHOB,FORCE_2020_LITHOFACIES_LITHOLOGY,WELL,RDEP_LOG10,RMED_LOG10,GR_MISSING,...,RHOB_ROLLSTD_9,RHOB_ROLLMEAN_21,RHOB_ROLLSTD_21,RESISTIVITY_SEPARATION,DEPT,GR_GRAD,RDEP_LOG10_GRAD,RMED_LOG10_GRAD,DTC_GRAD,RHOB_GRAD
73674,103.451515,0.878615,NaN,147.043427,NaN,65000.0,15_9-23,-0.056201,NaN,0,...,NaN,NaN,NaN,NaN,1518.2800,-61.359355,-0.014275,NaN,-10.637886,NaN
73675,94.124893,0.874237,NaN,145.426468,NaN,65000.0,15_9-23,-0.058371,NaN,0,...,NaN,NaN,NaN,NaN,1518.4320,-3.542147,-0.014346,NaN,-0.864230,NaN
73676,93.586487,0.869858,NaN,145.295105,NaN,65000.0,15_9-23,-0.060552,NaN,0,...,NaN,NaN,NaN,NaN,1518.5840,-16.270487,-0.014419,NaN,-10.773107,NaN
73677,91.113373,0.865479,NaN,143.657593,NaN,65000.0,15_9-23,-0.062743,NaN,0,...,NaN,NaN,NaN,NaN,1518.7360,59.966840,-0.005536,NaN,-5.389163,NaN
73678,100.228333,0.863804,0.849849,142.838440,NaN,65000.0,15_9-23,-0.063585,-0.070658,0,...,NaN,NaN,NaN,0.007073,1518.8880,-18.015962,0.060533,0.257131,-3.000360,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1404784,90.720284,3.158570,2.820439,75.260658,NaN,65000.0,35_9-7,0.499491,0.450317,0,...,NaN,NaN,NaN,0.049174,2973.2988,-24.067477,0.153565,0.196066,-2.581295,NaN
1404785,87.062027,3.332977,3.020778,74.868301,NaN,65000.0,35_9-7,0.522832,0.480119,0,...,NaN,NaN,NaN,0.042713,2973.4508,-6.224381,-0.258962,-0.221228,-0.132762,NaN
1404786,86.115921,3.044179,2.795711,74.848122,NaN,65000.0,35_9-7,0.483470,0.446492,0,...,NaN,NaN,NaN,0.036978,2973.6028,22.244805,-0.190650,-0.143578,0.762538,NaN
1404787,89.497131,2.847681,2.658694,74.964027,NaN,65000.0,35_9-7,0.454491,0.424668,0,...,NaN,NaN,NaN,0.029823,2973.7548,71.849472,0.122568,0.150706,0.969586,NaN


In [26]:
# ============================================================
# W5 — FINAL BLIND EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# True labels
# ------------------------------------------------------------
y_w5_blind = blind_df["FORCE_2020_LITHOFACIES_LITHOLOGY"].to_numpy()

# Saved W5 already predicts FORCE lithology codes
y_w5_pred = w5_blind_pred_raw.astype(int)

print("=" * 70)
print("W5 SAVED MODEL — BLIND EVALUATION")
print("=" * 70)

print(f"\nSamples: {len(y_w5_blind):,}")

# ------------------------------------------------------------
# Standard ML metrics
# ------------------------------------------------------------
w5_blind_accuracy = accuracy_score(
    y_w5_blind,
    y_w5_pred
)

w5_blind_balanced = balanced_accuracy_score(
    y_w5_blind,
    y_w5_pred
)

w5_blind_macro_f1 = f1_score(
    y_w5_blind,
    y_w5_pred,
    average="macro",
    zero_division=0
)

w5_blind_weighted_f1 = f1_score(
    y_w5_blind,
    y_w5_pred,
    average="weighted",
    zero_division=0
)

print("\nSTANDARD METRICS")
print("-" * 70)
print(f"Accuracy:          {w5_blind_accuracy:.6f}")
print(f"Balanced Accuracy: {w5_blind_balanced:.6f}")
print(f"Macro F1:          {w5_blind_macro_f1:.6f}")
print(f"Weighted F1:       {w5_blind_weighted_f1:.6f}")

# ------------------------------------------------------------
# Per-class report
# ------------------------------------------------------------
print("\nCLASSIFICATION REPORT")
print("-" * 70)

print(
    classification_report(
        y_w5_blind,
        y_w5_pred,
        labels=force_classes,
        target_names=[str(x) for x in force_classes],
        zero_division=0,
        digits=4
    )
)

# ------------------------------------------------------------
# Official FORCE score
# ------------------------------------------------------------
true_idx = np.array([
    force_to_index[int(y)]
    for y in y_w5_blind
])

pred_idx = np.array([
    force_to_index[int(y)]
    for y in y_w5_pred
])

penalties = penalty_matrix[
    true_idx,
    pred_idx
]

w5_mean_penalty = np.mean(penalties)
w5_force_score = -w5_mean_penalty

print("\nOFFICIAL FORCE METRIC")
print("-" * 70)
print(f"Mean penalty: {w5_mean_penalty:.6f}")
print(f"FORCE score:  {w5_force_score:.6f}")

print("\n" + "=" * 70)

W5 SAVED MODEL — BLIND EVALUATION

Samples: 122,576

STANDARD METRICS
----------------------------------------------------------------------
Accuracy:          0.738676
Balanced Accuracy: 0.463725
Macro F1:          0.433924
Weighted F1:       0.713718

CLASSIFICATION REPORT
----------------------------------------------------------------------
              precision    recall  f1-score   support

     30000.0     0.6007    0.5262    0.5610     14218
     65000.0     0.8208    0.9547    0.8827     71827
     65030.0     0.3652    0.2187    0.2736     12283
     70000.0     0.5687    0.5004    0.5323      8380
     70032.0     0.5947    0.2303    0.3320      2905
     74000.0     0.6400    0.0557    0.1026       287
     80000.0     0.1503    0.1126    0.1287      4396
     86000.0     0.7682    0.7052    0.7354       597
     88000.0     0.9911    0.8872    0.9363      6498
     90000.0     0.5602    0.8770    0.6837       244
     93000.0     0.0000    0.0000    0.0000         0
    

/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


NameError: name 'force_to_index' is not defined

In [29]:
# ============================================================
# W5 — OFFICIAL FORCE SCORE
# ============================================================

# Official FORCE class order
force_classes = np.array([
    30000, 65030, 65000, 80000,
    74000, 70000, 70032, 88000,
    86000, 99000, 90000, 93000
])

# Build code -> index mapping
force_to_index = {
    int(code): idx
    for idx, code in enumerate(force_classes)
}

# Convert true/predicted FORCE codes to matrix indices
true_idx = np.array([
    force_to_index[int(y)]
    for y in y_w5_blind
])

pred_idx = np.array([
    force_to_index[int(y)]
    for y in y_w5_pred
])

# Official geological penalty
w5_penalties = penalty_matrix[
    true_idx,
    pred_idx
]

w5_mean_penalty = np.mean(w5_penalties)
w5_force_score = -w5_mean_penalty

print("=" * 70)
print("W5 — OFFICIAL FORCE BLIND SCORE")
print("=" * 70)

print(f"\nSamples:      {len(y_w5_blind):,}")
print(f"Mean penalty: {w5_mean_penalty:.6f}")
print(f"FORCE score:  {w5_force_score:.6f}")

print("=" * 70)

W5 — OFFICIAL FORCE BLIND SCORE

Samples:      122,576
Mean penalty: 0.874511
FORCE score:  -0.874511


In [28]:
# ============================================================
# W5 — OFFICIAL FORCE SCORE (SELF-CONTAINED)
# ============================================================

# Official FORCE class order
force_classes = np.array([
    30000, 65030, 65000, 80000,
    74000, 70000, 70032, 88000,
    86000, 99000, 90000, 93000
])

# Official FORCE geological penalty matrix
penalty_matrix = np.array([
    [0,    2,     3.5,   3,     3.75, 3.5,   3.5,   4,    4,    2.5,   3.875, 3.25],
    [2,    0,     2.375, 2.75,  4,    3.75,  3.75, 3.875, 4,    3,     3.75,  3],
    [3.5,  2.375, 0,     2,     3.5,  3.5,  3.75, 4,    3.25, 2.75,  3.25,  3],
    [3,    2.75,  2,     0,     2.5,  2,    2.25, 4,    4,    3.375, 3.75,  3.25],
    [3.75, 4,     3.5,   2.5,   0,    2.625,2.875,3.75, 3,    3,     4,     3.625],
    [3.5,  3.75,  3.5,   2,     2.625,0,   1.375,4,    3.75, 3.5,   4,     3.625],
    [3.5,  3.75,  3.75,  2.25,  2.875,1.375,0,   4,    3.75, 3.125, 4,     3.75],
    [4,    3.875, 4,     4,     3.75, 4,    4,     4,    2.75, 3.75,  3.75,  4],
    [4,    4,     4,     4,     3.25, 3.75, 3.75, 2.75, 0,    4,     4,     3.875],
    [2.5,  3,     2.75,  3.375,3,    3.5,   3.125,3.75, 4,    0,     2.5,   3.25],
    [3.875,3.75,  3.25,  3.75, 4,    4,    4,     3.75, 4,    2.5,   0,     4],
    [3.25, 3,     3,     3.25,  3.625,3.625,3.75, 4,    3.875,3.25,  4,     0]
])

# Code -> matrix index
force_to_index = {
    int(code): idx
    for idx, code in enumerate(force_classes)
}

# Convert true/predicted lithology codes to matrix indices
true_idx = np.array([
    force_to_index[int(y)]
    for y in y_w5_blind
])

pred_idx = np.array([
    force_to_index[int(y)]
    for y in y_w5_pred
])

# Calculate geological penalties
w5_penalties = penalty_matrix[
    true_idx,
    pred_idx
]

w5_mean_penalty = np.mean(w5_penalties)
w5_force_score = -w5_mean_penalty

print("=" * 70)
print("W5 — OFFICIAL FORCE BLIND SCORE")
print("=" * 70)

print(f"\nSamples:      {len(y_w5_blind):,}")
print(f"Mean penalty: {w5_mean_penalty:.6f}")
print(f"FORCE score:  {w5_force_score:.6f}")

print("=" * 70)

W5 — OFFICIAL FORCE BLIND SCORE

Samples:      122,576
Mean penalty: 0.874511
FORCE score:  -0.874511


| Metric | W5 ExtraTrees | Final XGBoost | Better |
|---|---:|---:|---|
| Accuracy | **73.87%** | 73.82% | 🟢 W5 |
| Balanced Accuracy | 46.37% | **46.60%** | 🟢 XGB |
| Macro F1 | 43.39% | **43.67%** | 🟢 XGB |
| Weighted F1 | **71.37%** | 71.35% | 🟢 W5 |
| **FORCE score** | **−0.874511** | −0.961986 | 🟢 **W5** |

In [30]:
# ============================================================
# W5 — FORCE PENALTY ERROR ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Confusion matrix in OFFICIAL FORCE class order
# ------------------------------------------------------------
cm_w5 = pd.crosstab(
    pd.Series(y_w5_blind, name="True"),
    pd.Series(y_w5_pred, name="Pred"),
    dropna=False
).reindex(
    index=force_classes,
    columns=force_classes,
    fill_value=0
)

# ------------------------------------------------------------
# Build every TRUE -> PREDICTED error pair
# ------------------------------------------------------------
error_rows = []

for i, true_code in enumerate(force_classes):
    for j, pred_code in enumerate(force_classes):

        count = int(cm_w5.loc[true_code, pred_code])

        # Ignore correct predictions
        if true_code == pred_code or count == 0:
            continue

        penalty = float(penalty_matrix[i, j])

        total_penalty = count * penalty

        error_rows.append({
            "True": int(true_code),
            "Predicted": int(pred_code),
            "Count": count,
            "Penalty_per_sample": penalty,
            "Total_penalty": total_penalty
        })

error_df = pd.DataFrame(error_rows)

# ------------------------------------------------------------
# Percentage of TOTAL FORCE penalty
# ------------------------------------------------------------
total_penalty = error_df["Total_penalty"].sum()

error_df["Penalty_%"] = (
    error_df["Total_penalty"]
    / total_penalty
    * 100
)

# Sort by total contribution
error_df = error_df.sort_values(
    "Total_penalty",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Cumulative contribution
# ------------------------------------------------------------
error_df["Cumulative_%"] = (
    error_df["Penalty_%"].cumsum()
)

# ------------------------------------------------------------
# Print TOP 15
# ------------------------------------------------------------
print("=" * 90)
print("W5 — TOP FORCE PENALTY ERROR PAIRS")
print("=" * 90)

print(
    error_df.head(15).to_string(
        index=False,
        formatters={
            "Penalty_per_sample": "{:.3f}".format,
            "Total_penalty": "{:.1f}".format,
            "Penalty_%": "{:.2f}%".format,
            "Cumulative_%": "{:.2f}%".format,
        }
    )
)

print("\n" + "=" * 90)
print(f"Total FORCE penalty: {total_penalty:,.2f}")
print(f"Mean penalty:        {w5_mean_penalty:.6f}")
print(f"FORCE score:         {w5_force_score:.6f}")
print("=" * 90)

W5 — TOP FORCE PENALTY ERROR PAIRS
 True  Predicted  Count Penalty_per_sample Total_penalty Penalty_% Cumulative_%
65030      65000   7579              2.375       18000.1    21.39%       21.39%
30000      65000   2707              3.500        9474.5    11.26%       32.66%
70000      65000   1558              3.500        5453.0     6.48%       39.14%
70000      30000   1525              3.500        5337.5     6.34%       45.48%
80000      65000   2190              2.000        4380.0     5.21%       50.69%
30000      80000   1344              3.000        4032.0     4.79%       55.48%
30000      65030   2002              2.000        4004.0     4.76%       60.24%
65000      65030   1636              2.375        3885.5     4.62%       64.86%
65030      30000   1560              2.000        3120.0     3.71%       68.57%
88000      30000    648              4.000        2592.0     3.08%       71.65%
70032      70000   1689              1.375        2322.4     2.76%       74.41%
65000

freeze W5 as the production ML model


Our evidence is sufficient:
- W5 ExtraTrees: FORCE −0.874511
- Final XGBoost: FORCE −0.961986
- W5 is better on the project-specific metric.
- We have already investigated the major weakness: the 30000/65000/65030 boundary.
- The targeted error analysis shows where the penalty comes from.
- The 65030 specialist/hierarchical experiments did not improve the complete 12-class system.

Model: ExtraTreesClassifier

-n_estimators      = 100

-max_depth         = 20

-min_samples_leaf  = 5

-class_weight      = None

-n_jobs             = -1

-random_state       = 42

In [ ]:
# Training wells:    98
# Training samples:  1,171,557
# Features:          41
# Classes:           12

# # frozen schemas

# 1   GR
# 2   RDEP_LOG10
# 3   RMED_LOG10
# 4   DTC
# 5   RHOB

# 6   GR_ROLLMEAN_5
# 7   GR_ROLLSTD_5
# 8   GR_ROLLMEAN_9
# 9   GR_ROLLSTD_9
# 10  GR_ROLLMEAN_21
# 11  GR_ROLLSTD_21

# 12  RDEP_LOG10_ROLLMEAN_5
# 13  RDEP_LOG10_ROLLSTD_5
# 14  RDEP_LOG10_ROLLMEAN_9
# 15  RDEP_LOG10_ROLLSTD_9
# 16  RDEP_LOG10_ROLLMEAN_21
# 17  RDEP_LOG10_ROLLSTD_21

# 18  RMED_LOG10_ROLLMEAN_5
# 19  RMED_LOG10_ROLLSTD_5
# 20  RMED_LOG10_ROLLMEAN_9
# 21  RMED_LOG10_ROLLSTD_9
# 22  RMED_LOG10_ROLLMEAN_21
# 23  RMED_LOG10_ROLLSTD_21

# 24  DTC_ROLLMEAN_5
# 25  DTC_ROLLSTD_5
# 26  DTC_ROLLMEAN_9
# 27  DTC_ROLLSTD_9
# 28  DTC_ROLLMEAN_21
# 29  DTC_ROLLSTD_21

# 30  RHOB_ROLLMEAN_5
# 31  RHOB_ROLLSTD_5
# 32  RHOB_ROLLMEAN_9
# 33  RHOB_ROLLSTD_9
# 34  RHOB_ROLLMEAN_21
# 35  RHOB_ROLLSTD_21

# 36  RESISTIVITY_SEPARATION

# 37  GR_MISSING
# 38  RDEP_LOG10_MISSING
# 39  RMED_LOG10_MISSING
# 40  DTC_MISSING
# 41  RHOB_MISSING

In [32]:
# ============================================================
# FREEZE W5 — PRODUCTION MODEL MANIFEST
# ============================================================

import json
from pathlib import Path

manifest = {
    "model_name": "Lithology_W5_ExtraTrees",
    "status": "production_candidate_frozen",

    "algorithm": "ExtraTreesClassifier",

    "hyperparameters": {
        "n_estimators": 100,
        "max_depth": 20,
        "min_samples_leaf": 5,
        "class_weight": None,
        "n_jobs": -1,
        "random_state": 42
    },

    "training": {
        "official_split": "FORCE 2020 official TRAIN",
        "training_wells": 98,
        "training_samples": 1171557,
        "number_of_features": 41,
        "number_of_classes": 12
    },

    "features": w5_saved_feature_columns,

    "classes": [
        30000,
        65000,
        65030,
        70000,
        70032,
        74000,
        80000,
        86000,
        88000,
        90000,
        93000,
        99000
    ],

    "blind_evaluation": {
        "samples": 122576,
        "accuracy": 0.738676,
        "balanced_accuracy": 0.463725,
        "macro_f1": 0.433924,
        "weighted_f1": 0.713718,
        "force_score": -0.874511
    },

    "selection_reason": (
        "Selected as production model because it achieved a better "
        "official FORCE blind score than the final XGBoost model."
    ),

    "xgboost_blind_force_score": -0.961986
}

manifest_path = Path("../models/w5_production_manifest.json")

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("=" * 70)
print("W5 PRODUCTION MODEL FROZEN")
print("=" * 70)

print(f"\nManifest saved to:")
print(manifest_path)

print("\nModel:")
print(manifest["model_name"])

print("\nFeatures:", len(manifest["features"]))
print("Classes:", len(manifest["classes"]))

print("\nFORCE blind score:")
print(manifest["blind_evaluation"]["force_score"])

print("\nStatus:")
print(manifest["status"])

print("=" * 70)

W5 PRODUCTION MODEL FROZEN

Manifest saved to:
../models/w5_production_manifest.json

Model:
Lithology_W5_ExtraTrees

Features: 41
Classes: 12

FORCE blind score:
-0.874511

Status:
production_candidate_frozen


In [33]:
# ============================================================
# MODEL REGISTRY AUDIT
# ============================================================

from pathlib import Path

models_dir = Path("../models")

print("=" * 70)
print("SAVED MODEL FILES")
print("=" * 70)

if not models_dir.exists():
    print("ERROR: models directory does not exist.")
else:
    for path in sorted(models_dir.rglob("*")):
        if path.is_file():
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"{path.relative_to(models_dir)}  ({size_mb:.2f} MB)")

print("=" * 70)

SAVED MODEL FILES
boosted_label_mapping.json  (0.00 MB)
boosted_tune_results_lightgbm.json  (0.00 MB)
boosted_tune_results_xgboost.json  (0.00 MB)
boosted_tune_results_xgboost_noweight.json  (0.00 MB)
lightgbm_feature_metadata.json  (0.00 MB)
lithology_lightgbm_final.joblib  (37.19 MB)
lithology_w5_final.joblib  (428.86 MB)
lithology_xgboost_final.joblib  (47.39 MB)
lithology_xgboost_noweight_final.joblib  (28.31 MB)
preprocessing_params_final.json  (0.00 MB)
w5_class_weights.json  (0.00 MB)
w5_feature_metadata.json  (0.00 MB)
w5_production_manifest.json  (0.00 MB)
xgboost_feature_metadata.json  (0.00 MB)
xgboost_noweight_feature_metadata.json  (0.00 MB)


In [34]:
# ============================================================
# INSPECT SAVED MODEL METADATA
# ============================================================

import json
from pathlib import Path

metadata_files = [
    "w5_feature_metadata.json",
    "xgboost_feature_metadata.json",
    "lightgbm_feature_metadata.json",
]

print("=" * 80)
print("SAVED MODEL METADATA")
print("=" * 80)

for filename in metadata_files:

    path = Path("../models") / filename

    print(f"\n{'=' * 80}")
    print(filename)
    print("=" * 80)

    with open(path, "r") as f:
        metadata = json.load(f)

    # Print the complete JSON because these files are small
    print(json.dumps(metadata, indent=2))

SAVED MODEL METADATA

w5_feature_metadata.json
{
  "n_features": 41,
  "features": [
    "GR",
    "RDEP_LOG10",
    "RMED_LOG10",
    "DTC",
    "RHOB",
    "GR_ROLLMEAN_5",
    "GR_ROLLSTD_5",
    "GR_ROLLMEAN_9",
    "GR_ROLLSTD_9",
    "GR_ROLLMEAN_21",
    "GR_ROLLSTD_21",
    "RDEP_LOG10_ROLLMEAN_5",
    "RDEP_LOG10_ROLLSTD_5",
    "RDEP_LOG10_ROLLMEAN_9",
    "RDEP_LOG10_ROLLSTD_9",
    "RDEP_LOG10_ROLLMEAN_21",
    "RDEP_LOG10_ROLLSTD_21",
    "RMED_LOG10_ROLLMEAN_5",
    "RMED_LOG10_ROLLSTD_5",
    "RMED_LOG10_ROLLMEAN_9",
    "RMED_LOG10_ROLLSTD_9",
    "RMED_LOG10_ROLLMEAN_21",
    "RMED_LOG10_ROLLSTD_21",
    "DTC_ROLLMEAN_5",
    "DTC_ROLLSTD_5",
    "DTC_ROLLMEAN_9",
    "DTC_ROLLSTD_9",
    "DTC_ROLLMEAN_21",
    "DTC_ROLLSTD_21",
    "RHOB_ROLLMEAN_5",
    "RHOB_ROLLSTD_5",
    "RHOB_ROLLMEAN_9",
    "RHOB_ROLLSTD_9",
    "RHOB_ROLLMEAN_21",
    "RHOB_ROLLSTD_21",
    "RESISTIVITY_SEPARATION",
    "GR_MISSING",
    "RDEP_LOG10_MISSING",
    "RMED_LOG10_MISSING",
    "DT

In [36]:
# ============================================================
# VERIFY ALL THREE SAVED PRODUCTION MODELS
# ============================================================

import joblib
import numpy as np

model_paths = {
    "W5 ExtraTrees": "../models/lithology_w5_final.joblib",
    "XGBoost Tuned": "../models/lithology_xgboost_final.joblib",
    "LightGBM Tuned": "../models/lithology_lightgbm_final.joblib",
}

loaded_models = {}

print("=" * 80)
print("VERIFYING SAVED MODELS")
print("=" * 80)

for name, path in model_paths.items():

    print(f"\n{'-' * 80}")
    print(name)
    print("-" * 80)

    model = joblib.load(path)
    loaded_models[name] = model

    print("Type:", type(model).__name__)

    print("Classes:")
    print(model.classes_)

    if hasattr(model, "feature_names_in_"):
        print(
            "Number of model features:",
            len(model.feature_names_in_)
        )
        print(
            "First 5 features:",
            list(model.feature_names_in_[:])
        )
        print(
            "Last 5 features:",
            list(model.feature_names_in_[:])
        )
    else:
        print("feature_names_in_: NOT AVAILABLE")

print("\n" + "=" * 80)
print("ALL THREE MODELS LOADED SUCCESSFULLY")
print("=" * 80)

VERIFYING SAVED MODELS

--------------------------------------------------------------------------------
W5 ExtraTrees
--------------------------------------------------------------------------------
Type: ExtraTreesClassifier
Classes:
[30000. 65000. 65030. 70000. 70032. 74000. 80000. 86000. 88000. 90000.
 93000. 99000.]
Number of model features: 41
First 5 features: ['GR', 'RDEP_LOG10', 'RMED_LOG10', 'DTC', 'RHOB', 'GR_ROLLMEAN_5', 'GR_ROLLSTD_5', 'GR_ROLLMEAN_9', 'GR_ROLLSTD_9', 'GR_ROLLMEAN_21', 'GR_ROLLSTD_21', 'RDEP_LOG10_ROLLMEAN_5', 'RDEP_LOG10_ROLLSTD_5', 'RDEP_LOG10_ROLLMEAN_9', 'RDEP_LOG10_ROLLSTD_9', 'RDEP_LOG10_ROLLMEAN_21', 'RDEP_LOG10_ROLLSTD_21', 'RMED_LOG10_ROLLMEAN_5', 'RMED_LOG10_ROLLSTD_5', 'RMED_LOG10_ROLLMEAN_9', 'RMED_LOG10_ROLLSTD_9', 'RMED_LOG10_ROLLMEAN_21', 'RMED_LOG10_ROLLSTD_21', 'DTC_ROLLMEAN_5', 'DTC_ROLLSTD_5', 'DTC_ROLLMEAN_9', 'DTC_ROLLSTD_9', 'DTC_ROLLMEAN_21', 'DTC_ROLLSTD_21', 'RHOB_ROLLMEAN_5', 'RHOB_ROLLSTD_5', 'RHOB_ROLLMEAN_9', 'RHOB_ROLLSTD_9', 

In [37]:
# ============================================================
# LIGHTGBM — SAVED MODEL BLIND EVALUATION
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)
import numpy as np

# ------------------------------------------------------------
# Official FORCE class order used by the encoded models
# ------------------------------------------------------------
force_classes = np.array([
    30000, 65030, 65000, 80000,
    74000, 70000, 70032, 88000,
    86000, 99000, 90000, 93000
])

# ------------------------------------------------------------
# Use the exact feature order stored in the LightGBM model
# ------------------------------------------------------------
lgbm_model = loaded_models["LightGBM Tuned"]

lgbm_features = list(lgbm_model.feature_names_in_)

X_lgbm_blind = blind_df[lgbm_features]

print("=" * 70)
print("LIGHTGBM — BLIND PREDICTION")
print("=" * 70)

print(f"\nBLIND matrix: {X_lgbm_blind.shape}")
assert X_lgbm_blind.shape == (122576, 41)

# ------------------------------------------------------------
# Predict encoded classes
# ------------------------------------------------------------
lgbm_pred_encoded = lgbm_model.predict(X_lgbm_blind)

# LightGBM predictions are 0–11
lgbm_pred_encoded = np.asarray(
    lgbm_pred_encoded,
    dtype=int
)

# ------------------------------------------------------------
# Convert encoded classes → FORCE lithology codes
# ------------------------------------------------------------
lgbm_blind_pred = force_classes[lgbm_pred_encoded]

# ------------------------------------------------------------
# True labels
# ------------------------------------------------------------
y_lgbm_blind = y_w5_blind.astype(int)

# ------------------------------------------------------------
# Standard metrics
# ------------------------------------------------------------
lgbm_accuracy = accuracy_score(
    y_lgbm_blind,
    lgbm_blind_pred
)

lgbm_balanced = balanced_accuracy_score(
    y_lgbm_blind,
    lgbm_blind_pred
)

lgbm_macro_f1 = f1_score(
    y_lgbm_blind,
    lgbm_blind_pred,
    average="macro",
    zero_division=0
)

lgbm_weighted_f1 = f1_score(
    y_lgbm_blind,
    lgbm_blind_pred,
    average="weighted",
    zero_division=0
)

print("\nSTANDARD METRICS")
print("-" * 70)
print(f"Accuracy:          {lgbm_accuracy:.6f}")
print(f"Balanced Accuracy: {lgbm_balanced:.6f}")
print(f"Macro F1:          {lgbm_macro_f1:.6f}")
print(f"Weighted F1:       {lgbm_weighted_f1:.6f}")

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------
print("\nCLASSIFICATION REPORT")
print("-" * 70)

print(
    classification_report(
        y_lgbm_blind,
        lgbm_blind_pred,
        labels=force_classes,
        target_names=[str(x) for x in force_classes],
        zero_division=0,
        digits=4
    )
)

print("=" * 70)

LIGHTGBM — BLIND PREDICTION

BLIND matrix: (122576, 41)

STANDARD METRICS
----------------------------------------------------------------------
Accuracy:          0.160260
Balanced Accuracy: 0.135393
Macro F1:          0.082675
Weighted F1:       0.149020

CLASSIFICATION REPORT
----------------------------------------------------------------------
              precision    recall  f1-score   support

       30000     0.5885    0.5216    0.5530     14218
       65030     0.0891    0.5384    0.1528     12283
       65000     0.4219    0.0629    0.1094     71827
       80000     0.1046    0.2316    0.1441      4396
       74000     0.0071    0.0348    0.0118       287
       70000     0.1892    0.0008    0.0017      8380
       70032     0.0010    0.0021    0.0014      2905
       88000     0.0000    0.0000    0.0000      6498
       86000     0.0098    0.0972    0.0178       597
       99000     0.0000    0.0000    0.0000       941
       90000     0.0000    0.0000    0.0000       244


/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [38]:
# ============================================================
# CHECK SAVED BOOSTED-MODEL LABEL MAPPING
# ============================================================

import json

mapping_path = "../models/boosted_label_mapping.json"

with open(mapping_path, "r") as f:
    boosted_mapping = json.load(f)

print("=" * 70)
print("SAVED BOOSTED MODEL LABEL MAPPING")
print("=" * 70)

print(json.dumps(boosted_mapping, indent=2))

print("=" * 70)

SAVED BOOSTED MODEL LABEL MAPPING
{
  "code_to_idx": {
    "30000": 0,
    "65000": 1,
    "65030": 2,
    "70000": 3,
    "70032": 4,
    "74000": 5,
    "80000": 6,
    "86000": 7,
    "88000": 8,
    "90000": 9,
    "93000": 10,
    "99000": 11
  },
  "idx_to_code": {
    "0": 30000,
    "1": 65000,
    "2": 65030,
    "3": 70000,
    "4": 70032,
    "5": 74000,
    "6": 80000,
    "7": 86000,
    "8": 88000,
    "9": 90000,
    "10": 93000,
    "11": 99000
  }
}


In [39]:
# ============================================================
# LIGHTGBM — CORRECT LABEL DECODING
# ============================================================

# Saved boosted-model mapping
idx_to_code = {
    0: 30000,
    1: 65000,
    2: 65030,
    3: 70000,
    4: 70032,
    5: 74000,
    6: 80000,
    7: 86000,
    8: 88000,
    9: 90000,
    10: 93000,
    11: 99000,
}

# Correct encoded index → FORCE lithology code
lgbm_blind_pred = np.array([
    idx_to_code[int(idx)]
    for idx in lgbm_pred_encoded
])

# True labels
y_lgbm_blind = y_w5_blind.astype(int)

# ------------------------------------------------------------
# Standard metrics
# ------------------------------------------------------------

lgbm_accuracy = accuracy_score(
    y_lgbm_blind,
    lgbm_blind_pred
)

lgbm_balanced = balanced_accuracy_score(
    y_lgbm_blind,
    lgbm_blind_pred
)

lgbm_macro_f1 = f1_score(
    y_lgbm_blind,
    lgbm_blind_pred,
    average="macro",
    zero_division=0
)

lgbm_weighted_f1 = f1_score(
    y_lgbm_blind,
    lgbm_blind_pred,
    average="weighted",
    zero_division=0
)

print("=" * 70)
print("LIGHTGBM — CORRECTED BLIND EVALUATION")
print("=" * 70)

print(f"\nSamples:            {len(y_lgbm_blind):,}")
print(f"Accuracy:           {lgbm_accuracy:.6f}")
print(f"Balanced Accuracy:  {lgbm_balanced:.6f}")
print(f"Macro F1:           {lgbm_macro_f1:.6f}")
print(f"Weighted F1:        {lgbm_weighted_f1:.6f}")

print("\nClassification Report")
print("-" * 70)

print(
    classification_report(
        y_lgbm_blind,
        lgbm_blind_pred,
        labels=force_classes,
        target_names=[str(x) for x in force_classes],
        zero_division=0,
        digits=4
    )
)

print("=" * 70)

LIGHTGBM — CORRECTED BLIND EVALUATION

Samples:            122,576
Accuracy:           0.705252
Balanced Accuracy:  0.482206
Macro F1:           0.451275
Weighted F1:        0.703439

Classification Report
----------------------------------------------------------------------
              precision    recall  f1-score   support

       30000     0.5885    0.5216    0.5530     14218
       65030     0.3155    0.2749    0.2938     12283
       65000     0.8351    0.8633    0.8490     71827
       80000     0.1552    0.2097    0.1784      4396
       74000     0.0270    0.0035    0.0062       287
       70000     0.5288    0.6143    0.5684      8380
       70032     0.7666    0.3697    0.4988      2905
       88000     0.9751    0.8870    0.9290      6498
       86000     0.7859    0.6332    0.7013       597
       99000     0.1489    0.1934    0.1683       941
       90000     0.6151    0.7336    0.6692       244
       93000     0.0000    0.0000    0.0000         0

    accuracy       

/Users/akashray/Lithology_Predictor/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [40]:
# ============================================================
# LIGHTGBM — OFFICIAL FORCE BLIND SCORE
# ============================================================

# force_classes and penalty_matrix were already defined
# in the previous FORCE-score cell.

true_idx_lgbm = np.array([
    force_to_index[int(y)]
    for y in y_lgbm_blind
])

pred_idx_lgbm = np.array([
    force_to_index[int(y)]
    for y in lgbm_blind_pred
])

lgbm_penalties = penalty_matrix[
    true_idx_lgbm,
    pred_idx_lgbm
]

lgbm_mean_penalty = np.mean(lgbm_penalties)
lgbm_force_score = -lgbm_mean_penalty

print("=" * 70)
print("LIGHTGBM — OFFICIAL FORCE BLIND SCORE")
print("=" * 70)

print(f"\nSamples:      {len(y_lgbm_blind):,}")
print(f"Mean penalty: {lgbm_mean_penalty:.6f}")
print(f"FORCE score:  {lgbm_force_score:.6f}")

print("=" * 70)

LIGHTGBM — OFFICIAL FORCE BLIND SCORE

Samples:      122,576
Mean penalty: 0.970091
FORCE score:  -0.970091


| Model | Accuracy | Balanced Accuracy | Macro F1 | Weighted F1 | **FORCE Score** |
|---|---:|---:|---:|---:|---:|
| 🥇 **W5 ExtraTrees** | **73.87%** | 46.37% | 43.39% | **71.37%** | **−0.874511** |
| 🥈 XGBoost Tuned | 73.82% | 46.60% | 43.67% | 71.35% | −0.961986 |
| 🥉 LightGBM Tuned | 70.53% | **48.22%** | **45.13%** | 70.34% | −0.970091 |

end of the models selction and improvemnt methods...now going to use these three models which is performed bwtter in all the feature experiments.